# 🍎🍌🍊 Práctica 4 - Parte 1A: Segmentación de Frutas con YOLOv11

**Estudiante:** Andres Morocho  
**Asignatura:** Visión por Computador  
**Docente:** Ing. Vladimir Robles Bykbaev  
**Período:** Marzo – Agosto 2026  

---

## Objetivo
Aplicar **transfer learning** sobre la red YOLOv11 (Ultralytics) para realizar **segmentación de instancias** de tres clases de frutas:
- 🍎 Manzana
- 🍌 Banana
- 🍊 Naranja

Se empleará un dataset personalizado obtenido de Roboflow Universe y se compararán los resultados de inferencia en **GPU vs CPU**, midiendo cuadros por segundo (FPS).

---

## Herramientas utilizadas
- **Ultralytics YOLO** (segmentación de instancias)
- **Roboflow** (dataset con anotaciones de segmentación)
- **Google Colab** (entorno de ejecución con GPU)
- **OpenCV** (procesamiento de video e inferencia)

### Cita
> Este notebook utiliza la librería [Ultralytics](https://github.com/ultralytics/ultralytics) para entrenamiento y la plataforma [Roboflow](https://roboflow.com/) para gestión de datasets.  
> Jocher, G., Chaurasia, A., & Qiu, J. (2023). Ultralytics YOLO (Version 8.0.0). https://github.com/ultralytics/ultralytics

## 1. Configuración del Entorno
Instalamos las dependencias necesarias y verificamos que la GPU esté disponible.

In [ ]:
# ==============================================================================
# 1.1 Instalación de dependencias
# ==============================================================================
!pip install -q ultralytics roboflow opencv-python-headless matplotlib seaborn

import torch
import os
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Please enable GPU in Colab: Runtime > Change runtime type > T4 GPU")

In [ ]:
# ==============================================================================
# 1.2 Verificar GPU con nvidia-smi
# ==============================================================================
!nvidia-smi

## 2. Descarga del Dataset desde Roboflow

Utilizamos un dataset de segmentación de instancias de frutas (Apple, Banana, Orange) de Roboflow Universe.  
**Fuente del dataset:** [Roboflow Universe - Fruit Instance Segmentation](https://universe.roboflow.com/)

> **Nota:** Debes obtener tu propia API key de Roboflow (gratuita) y reemplazarla en la celda siguiente.  
> Para ello: Crea cuenta en roboflow.com → Settings → API Key

### Pasos para obtener el dataset:
1. Ir a https://universe.roboflow.com/
2. Buscar "fruit instance segmentation apple banana orange"
3. Seleccionar un dataset con anotaciones de **Instance Segmentation**
4. Exportar en formato **YOLOv11**
5. Copiar el código de descarga que Roboflow proporciona

In [ ]:
# ==============================================================================
# 2.1 Descargar dataset de Roboflow
# ==============================================================================
# INSTRUCCIONES:
# 1. Ve a https://universe.roboflow.com/ y busca un dataset de frutas con
#    segmentación de instancias (apple, banana, orange).
# 2. Ejemplo recomendado:
#    https://universe.roboflow.com/fruitsdetection/instance-segmentation-of-fruit-images
# 3. Click en "Download Dataset" > Formato "YOLOv11" > "Show download code"
# 4. Copia el snippet de código que te da Roboflow y pégalo aquí abajo,
#    reemplazando el bloque de ejemplo.

from roboflow import Roboflow

# ==================== REEMPLAZAR CON TU CÓDIGO DE ROBOFLOW ====================
# Ejemplo (debes reemplazar con tu API key y tu proyecto/versión real):

rf = Roboflow(api_key="TU_API_KEY_AQUI")  # <-- Reemplaza con tu API key
project = rf.workspace("fruitsdetection").project("instance-segmentation-of-fruit-images")
version = project.version(1)
dataset = version.download("yolov11")

# =============================================================================

print(f"\n✅ Dataset descargado en: {dataset.location}")

In [ ]:
# ==============================================================================
# 2.2 Explorar la estructura del dataset
# ==============================================================================
import yaml

# Leer el archivo data.yaml
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("📁 Configuración del dataset:")
print(f"   Clases: {data_config.get('names', 'N/A')}")
print(f"   Número de clases: {data_config.get('nc', 'N/A')}")

# Contar imágenes por split
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(dataset.location, split, 'images')
    if os.path.exists(split_path):
        n_images = len(os.listdir(split_path))
        print(f"   {split}: {n_images} imágenes")
    else:
        print(f"   {split}: No encontrado")

In [ ]:
# ==============================================================================
# 2.3 Visualizar muestras del dataset
# ==============================================================================
from PIL import Image
import glob

train_images_dir = os.path.join(dataset.location, 'train', 'images')
sample_images = sorted(glob.glob(os.path.join(train_images_dir, '*')))[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Muestras del Dataset de Entrenamiento', fontsize=16, fontweight='bold')

for i, img_path in enumerate(sample_images):
    ax = axes[i // 4][i % 4]
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(img_path)[:20], fontsize=8)
    ax.axis('off')

# Ocultar ejes vacíos
for i in range(len(sample_images), 8):
    axes[i // 4][i % 4].axis('off')

plt.tight_layout()
plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Muestras guardadas en dataset_samples.png")

## 3. Entrenamiento del Modelo (Transfer Learning)

Realizamos **fine-tuning** sobre el modelo pre-entrenado `yolo11n-seg.pt` (nano, para velocidad).  
El modelo base ya fue entrenado sobre el dataset COCO y tiene conocimiento general de objetos; nosotros lo adaptamos a nuestras 3 clases de frutas.

### Parámetros clave:
- **model**: `yolo11n-seg.pt` (modelo nano con segmentación de instancias)
- **epochs**: 50 (ajustable según resultados)
- **imgsz**: 640 (resolución estándar)
- **batch**: 16 (ajustar según VRAM disponible)
- **device**: 0 (GPU)

In [ ]:
# ==============================================================================
# 3.1 Entrenamiento con YOLOv11 (Transfer Learning / Fine-tuning)
# ==============================================================================
from ultralytics import YOLO

# Cargar modelo pre-entrenado de segmentación (transfer learning)
model = YOLO("yolo11n-seg.pt")

# Entrenar con fine-tuning sobre nuestro dataset de frutas
results = model.train(
    data=data_yaml_path,
    epochs=50,              # Número de épocas
    imgsz=640,              # Tamaño de imagen
    batch=16,               # Batch size (reducir a 8 si no hay suficiente VRAM)
    device=0,               # GPU (usar 'cpu' para CPU)
    project="runs/segment", # Directorio de salida
    name="fruit_seg",       # Nombre del experimento
    patience=10,            # Early stopping
    save=True,              # Guardar checkpoints
    plots=True,             # Generar gráficas de entrenamiento
    verbose=True,
)

print("\n✅ Entrenamiento completado.")
print(f"   Mejor modelo guardado en: runs/segment/fruit_seg/weights/best.pt")

In [ ]:
# ==============================================================================
# 3.2 Visualizar curvas de entrenamiento
# ==============================================================================
# YOLO genera automáticamente las curvas en el directorio del experimento
results_dir = Path("runs/segment/fruit_seg")

# Mostrar la imagen de resultados generada por YOLO
results_img = results_dir / "results.png"
if results_img.exists():
    img = Image.open(results_img)
    plt.figure(figsize=(18, 8))
    plt.imshow(img)
    plt.title('Curvas de Entrenamiento - YOLOv11 Fruit Segmentation', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No se encontró results.png. Verifica el directorio del experimento.")
    print(f"   Buscando en: {results_dir}")
    # Listar archivos disponibles
    if results_dir.exists():
        for f in sorted(results_dir.iterdir()):
            print(f"   - {f.name}")

## 4. Validación y Métricas del Modelo

Evaluamos el modelo entrenado sobre el conjunto de validación para obtener las métricas de rendimiento:
- **mAP@50**: Mean Average Precision al 50% de IoU
- **mAP@50-95**: mAP promediado sobre múltiples umbrales de IoU
- **Precisión y Recall** por clase

In [ ]:
# ==============================================================================
# 4.1 Validación del modelo entrenado
# ==============================================================================
best_model_path = "runs/segment/fruit_seg/weights/best.pt"
model_trained = YOLO(best_model_path)

# Ejecutar validación
metrics = model_trained.val(
    data=data_yaml_path,
    device=0,
    plots=True,
    save_json=True,
)

# Mostrar métricas principales
print("\n" + "="*60)
print("📊 MÉTRICAS DE VALIDACIÓN")
print("="*60)
print(f"   Box mAP@50:      {metrics.box.map50:.4f}")
print(f"   Box mAP@50-95:   {metrics.box.map:.4f}")
print(f"   Mask mAP@50:     {metrics.seg.map50:.4f}")
print(f"   Mask mAP@50-95:  {metrics.seg.map:.4f}")
print("="*60)

In [ ]:
# ==============================================================================
# 4.2 Visualizar predicciones sobre el set de validación
# ==============================================================================
# YOLO genera predicciones visuales durante la validación
val_preds = results_dir / "val_batch0_pred.jpg"
val_labels = results_dir / "val_batch0_labels.jpg"

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Ground Truth
if val_labels.exists():
    img_gt = Image.open(val_labels)
    axes[0].imshow(img_gt)
    axes[0].set_title('Ground Truth (Etiquetas Reales)', fontsize=14, fontweight='bold')
    axes[0].axis('off')

# Predicciones
if val_preds.exists():
    img_pred = Image.open(val_preds)
    axes[1].imshow(img_pred)
    axes[1].set_title('Predicciones del Modelo', fontsize=14, fontweight='bold')
    axes[1].axis('off')

plt.suptitle('Comparación: Ground Truth vs Predicciones', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('validation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================================
# 4.3 Mostrar la Matriz de Confusión
# ==============================================================================
conf_matrix_path = results_dir / "confusion_matrix.png"
conf_matrix_norm_path = results_dir / "confusion_matrix_normalized.png"

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

if conf_matrix_path.exists():
    img_cm = Image.open(conf_matrix_path)
    axes[0].imshow(img_cm)
    axes[0].set_title('Matriz de Confusión', fontsize=12, fontweight='bold')
    axes[0].axis('off')

if conf_matrix_norm_path.exists():
    img_cm_norm = Image.open(conf_matrix_norm_path)
    axes[1].imshow(img_cm_norm)
    axes[1].set_title('Matriz de Confusión (Normalizada)', fontsize=12, fontweight='bold')
    axes[1].axis('off')

plt.suptitle('Matrices de Confusión del Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Inferencia sobre Imágenes de Prueba

Probamos el modelo entrenado sobre imágenes individuales para verificar la calidad de la segmentación.

In [ ]:
# ==============================================================================
# 5.1 Inferencia sobre imágenes del set de test/validación
# ==============================================================================
import random

# Buscar imágenes de prueba
test_dir = os.path.join(dataset.location, 'test', 'images')
if not os.path.exists(test_dir):
    test_dir = os.path.join(dataset.location, 'valid', 'images')

test_images = sorted(glob.glob(os.path.join(test_dir, '*')))

# Seleccionar 6 imágenes al azar
sample_test = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Resultados de Segmentación de Instancias (YOLOv11 fine-tuned)', 
             fontsize=16, fontweight='bold')

for idx, img_path in enumerate(sample_test):
    ax = axes[idx // 3][idx % 3]
    
    # Ejecutar inferencia
    results = model_trained.predict(
        source=img_path,
        conf=0.5,
        device=0,
        verbose=False
    )
    
    # Obtener la imagen anotada con las máscaras
    annotated = results[0].plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    ax.imshow(annotated_rgb)
    
    # Contar detecciones por clase
    if results[0].boxes is not None and len(results[0].boxes) > 0:
        classes_detected = results[0].boxes.cls.cpu().numpy()
        class_names = results[0].names
        det_summary = ", ".join([f"{class_names[int(c)]}" for c in classes_detected])
        ax.set_title(f'Detectado: {det_summary}', fontsize=10)
    else:
        ax.set_title('Sin detecciones', fontsize=10)
    ax.axis('off')

# Ocultar ejes vacíos
for i in range(len(sample_test), 6):
    axes[i // 3][i % 3].axis('off')

plt.tight_layout()
plt.savefig('inference_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Resultados de inferencia guardados en inference_results.png")

## 6. Comparación de Rendimiento: GPU vs CPU

Esta sección es **crucial para la rúbrica**. Comparamos el rendimiento del modelo al ejecutar inferencia en:
- **GPU** (CUDA)
- **CPU**

Medimos:
- Cuadros por Segundo (FPS)
- Tiempo promedio por frame
- Uso de memoria

In [ ]:
# ==============================================================================
# 6.1 Benchmark GPU vs CPU en imágenes
# ==============================================================================
import psutil

def benchmark_inference(model_path, images, device, num_warmup=5, num_runs=3):
    """
    Realiza benchmark de inferencia sobre un conjunto de imágenes.
    
    Args:
        model_path: Ruta al modelo .pt
        images: Lista de rutas de imágenes
        device: 'cpu' o 0 (GPU)
        num_warmup: Frames de calentamiento (warm-up)
        num_runs: Número de repeticiones del benchmark
    
    Returns:
        dict: Resultados del benchmark
    """
    model = YOLO(model_path)
    
    # Warm-up: las primeras inferencias son más lentas por carga en memoria
    print(f"   🔄 Warm-up ({num_warmup} frames)...")
    for img in images[:num_warmup]:
        model.predict(source=img, device=device, verbose=False)
    
    # Benchmark
    all_times = []
    
    for run in range(num_runs):
        run_times = []
        for img in images:
            start_time = time.perf_counter()
            model.predict(source=img, device=device, verbose=False, conf=0.5)
            end_time = time.perf_counter()
            run_times.append(end_time - start_time)
        all_times.extend(run_times)
    
    times = np.array(all_times)
    fps_values = 1.0 / times
    
    # Obtener uso de memoria
    ram_usage = psutil.Process().memory_info().rss / 1e6  # MB
    
    gpu_mem = None
    if device != 'cpu' and torch.cuda.is_available():
        gpu_mem = torch.cuda.memory_allocated() / 1e6  # MB
    
    return {
        'device': 'GPU (CUDA)' if device != 'cpu' else 'CPU',
        'avg_time': np.mean(times),
        'std_time': np.std(times),
        'min_time': np.min(times),
        'max_time': np.max(times),
        'avg_fps': np.mean(fps_values),
        'std_fps': np.std(fps_values),
        'ram_mb': ram_usage,
        'gpu_mem_mb': gpu_mem,
        'fps_values': fps_values,
        'times': times,
    }

# Seleccionar imágenes para benchmark
benchmark_images = test_images[:30]  # Usar 30 imágenes
print(f"🔬 Benchmark con {len(benchmark_images)} imágenes x 3 repeticiones\n")

# Benchmark en GPU
print("🟢 Ejecutando benchmark en GPU...")
gpu_results = benchmark_inference(best_model_path, benchmark_images, device=0)
print(f"   ✅ GPU: {gpu_results['avg_fps']:.1f} FPS (±{gpu_results['std_fps']:.1f})")

# Liberar memoria GPU antes de CPU test
torch.cuda.empty_cache()

# Benchmark en CPU
print("\n🔵 Ejecutando benchmark en CPU...")
cpu_results = benchmark_inference(best_model_path, benchmark_images, device='cpu')
print(f"   ✅ CPU: {cpu_results['avg_fps']:.1f} FPS (±{cpu_results['std_fps']:.1f})")

# Resumen
speedup = gpu_results['avg_fps'] / cpu_results['avg_fps']
print(f"\n{'='*60}")
print(f"📊 RESUMEN DE RENDIMIENTO")
print(f"{'='*60}")
print(f"   GPU FPS promedio: {gpu_results['avg_fps']:.2f}")
print(f"   CPU FPS promedio: {cpu_results['avg_fps']:.2f}")
print(f"   Speedup GPU/CPU:  {speedup:.2f}x")
print(f"   RAM (GPU mode):   {gpu_results['ram_mb']:.0f} MB")
print(f"   RAM (CPU mode):   {cpu_results['ram_mb']:.0f} MB")
if gpu_results['gpu_mem_mb']:
    print(f"   VRAM (GPU):       {gpu_results['gpu_mem_mb']:.0f} MB")
print(f"{'='*60}")

In [ ]:
# ==============================================================================
# 6.2 Gráfica comparativa GPU vs CPU
# ==============================================================================
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Comparación de Rendimiento: GPU vs CPU', fontsize=16, fontweight='bold')

colors_gpu = '#2ecc71'  # Verde
colors_cpu = '#e74c3c'  # Rojo

# --- Gráfico 1: Barras de FPS ---
ax1 = axes[0]
devices = ['GPU (CUDA)', 'CPU']
fps_means = [gpu_results['avg_fps'], cpu_results['avg_fps']]
fps_stds = [gpu_results['std_fps'], cpu_results['std_fps']]
bars = ax1.bar(devices, fps_means, yerr=fps_stds, 
               color=[colors_gpu, colors_cpu],
               edgecolor='black', linewidth=1.2, capsize=10, alpha=0.85)
ax1.set_ylabel('Cuadros por Segundo (FPS)', fontsize=12)
ax1.set_title('FPS Promedio', fontsize=13, fontweight='bold')
for bar, val in zip(bars, fps_means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=14)
ax1.grid(axis='y', alpha=0.3)

# --- Gráfico 2: Distribución de tiempos ---
ax2 = axes[1]
data_boxplot = [
    gpu_results['times'] * 1000,  # a milisegundos
    cpu_results['times'] * 1000,
]
bp = ax2.boxplot(data_boxplot, labels=devices, patch_artist=True,
                 boxprops=dict(linewidth=1.5),
                 whiskerprops=dict(linewidth=1.5),
                 medianprops=dict(linewidth=2, color='black'))
bp['boxes'][0].set_facecolor(colors_gpu)
bp['boxes'][1].set_facecolor(colors_cpu)
ax2.set_ylabel('Tiempo por Frame (ms)', fontsize=12)
ax2.set_title('Distribución de Tiempos de Inferencia', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# --- Gráfico 3: Uso de memoria ---
ax3 = axes[2]
mem_labels = ['RAM\n(GPU mode)', 'RAM\n(CPU mode)']
mem_values = [gpu_results['ram_mb'], cpu_results['ram_mb']]
mem_colors = [colors_gpu, colors_cpu]

if gpu_results['gpu_mem_mb']:
    mem_labels.append('VRAM\n(GPU)')
    mem_values.append(gpu_results['gpu_mem_mb'])
    mem_colors.append('#3498db')

bars_mem = ax3.bar(mem_labels, mem_values, color=mem_colors,
                   edgecolor='black', linewidth=1.2, alpha=0.85)
ax3.set_ylabel('Memoria (MB)', fontsize=12)
ax3.set_title('Uso de Memoria', fontsize=13, fontweight='bold')
for bar, val in zip(bars_mem, mem_values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{val:.0f} MB', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('gpu_vs_cpu_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráficas guardadas en gpu_vs_cpu_comparison.png")

In [ ]:
# ==============================================================================
# 6.3 Tabla resumen detallada
# ==============================================================================
import pandas as pd

summary_data = {
    'Métrica': [
        'FPS Promedio',
        'FPS Desviación Estándar',
        'Tiempo Promedio (ms)',
        'Tiempo Mínimo (ms)',
        'Tiempo Máximo (ms)',
        'RAM (MB)',
        'VRAM / GPU Memory (MB)',
        'Speedup vs CPU',
    ],
    'GPU (CUDA)': [
        f"{gpu_results['avg_fps']:.2f}",
        f"{gpu_results['std_fps']:.2f}",
        f"{gpu_results['avg_time']*1000:.2f}",
        f"{gpu_results['min_time']*1000:.2f}",
        f"{gpu_results['max_time']*1000:.2f}",
        f"{gpu_results['ram_mb']:.0f}",
        f"{gpu_results['gpu_mem_mb']:.0f}" if gpu_results['gpu_mem_mb'] else 'N/A',
        f"{speedup:.2f}x",
    ],
    'CPU': [
        f"{cpu_results['avg_fps']:.2f}",
        f"{cpu_results['std_fps']:.2f}",
        f"{cpu_results['avg_time']*1000:.2f}",
        f"{cpu_results['min_time']*1000:.2f}",
        f"{cpu_results['max_time']*1000:.2f}",
        f"{cpu_results['ram_mb']:.0f}",
        'N/A',
        '1.00x (baseline)',
    ],
}

df_summary = pd.DataFrame(summary_data)
print("\n📊 TABLA RESUMEN DE RENDIMIENTO GPU vs CPU")
print("="*70)
print(df_summary.to_string(index=False))
print("="*70)

## 7. Inferencia en Video (Simulación de Webcam)

Para simular la inferencia en tiempo real desde webcam (como pide la rúbrica), procesamos un video y mostramos los FPS en overlay.

> **Nota**: En Google Colab no hay webcam, pero este mismo código funciona en el laboratorio cambiando la fuente de video a `cv2.VideoCapture(0)` para la webcam real.

In [ ]:
# ==============================================================================
# 7.1 Descargar un video de prueba (frutas)
# ==============================================================================
# Si tienes tu propio video, cámbialo aquí.
# Para Colab, podemos generar un video de prueba a partir de las imágenes del dataset.

def create_test_video_from_images(image_paths, output_path, fps=15, resize=(640, 480)):
    """
    Crea un video de prueba a partir de imágenes del dataset.
    Esto simula un flujo de video para benchmark.
    """
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, resize)
    
    # Repetir imágenes para tener al menos 10 segundos de video
    total_frames_needed = fps * 10  # 10 segundos
    images_cycle = (image_paths * ((total_frames_needed // len(image_paths)) + 1))[:total_frames_needed]
    
    for img_path in images_cycle:
        frame = cv2.imread(img_path)
        if frame is not None:
            frame = cv2.resize(frame, resize)
            writer.write(frame)
    
    writer.release()
    print(f"✅ Video de prueba creado: {output_path} ({total_frames_needed} frames, {fps} FPS)")
    return output_path

video_path = create_test_video_from_images(test_images, 'test_fruit_video.mp4')

In [ ]:
# ==============================================================================
# 7.2 Procesar video con inferencia GPU y CPU, mostrando FPS en overlay
# ==============================================================================

def process_video_with_overlay(model_path, video_path, output_path, device, max_frames=100):
    """
    Procesa un video con el modelo YOLO, dibujando FPS, uso de memoria
    y dispositivo en overlay sobre cada frame.
    
    Este código es el que se ejecutará en el laboratorio con la webcam real.
    Para webcam: cambiar cv2.VideoCapture(video_path) por cv2.VideoCapture(0)
    """
    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)
    
    # Obtener propiedades del video
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    orig_fps = int(cap.get(cv2.CAP_PROP_FPS))
    
    # Configurar writer para video de salida
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, orig_fps, (width, height))
    
    device_name = 'GPU (CUDA)' if device != 'cpu' else 'CPU'
    fps_history = []
    frame_count = 0
    
    print(f"   Procesando video en {device_name}...")
    
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Medir tiempo de inferencia
        t_start = time.perf_counter()
        results = model.predict(source=frame, device=device, verbose=False, conf=0.5)
        t_end = time.perf_counter()
        
        inference_time = t_end - t_start
        current_fps = 1.0 / inference_time if inference_time > 0 else 0
        fps_history.append(current_fps)
        
        # Dibujar frame anotado con segmentación
        annotated = results[0].plot()
        
        # ===== OVERLAY DE INFORMACIÓN =====
        # Fondo semitransparente para el texto
        overlay = annotated.copy()
        cv2.rectangle(overlay, (10, 10), (380, 140), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, annotated, 0.4, 0, annotated)
        
        # Texto con información
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_title = (0, 255, 255)  # Amarillo
        color_value = (0, 255, 0) if device != 'cpu' else (0, 128, 255)  # Verde o Naranja
        
        cv2.putText(annotated, f'Dispositivo: {device_name}', (20, 40),
                    font, 0.7, color_title, 2)
        cv2.putText(annotated, f'FPS: {current_fps:.1f}', (20, 70),
                    font, 0.8, color_value, 2)
        cv2.putText(annotated, f'Tiempo: {inference_time*1000:.1f} ms', (20, 100),
                    font, 0.7, color_value, 2)
        
        # Número de detecciones
        n_dets = len(results[0].boxes) if results[0].boxes is not None else 0
        cv2.putText(annotated, f'Detecciones: {n_dets}', (20, 130),
                    font, 0.7, (255, 255, 255), 2)
        
        writer.write(annotated)
        frame_count += 1
    
    cap.release()
    writer.release()
    
    avg_fps = np.mean(fps_history) if fps_history else 0
    print(f"   ✅ Video procesado: {frame_count} frames, FPS promedio: {avg_fps:.1f}")
    
    return fps_history

# Procesar en GPU
print("🟢 Procesando video en GPU...")
fps_gpu_video = process_video_with_overlay(
    best_model_path, video_path, 'output_gpu.mp4', device=0
)

torch.cuda.empty_cache()

# Procesar en CPU
print("\n🔵 Procesando video en CPU...")
fps_cpu_video = process_video_with_overlay(
    best_model_path, video_path, 'output_cpu.mp4', device='cpu'
)

In [ ]:
# ==============================================================================
# 7.3 Gráfica de FPS a lo largo del video
# ==============================================================================
fig, ax = plt.subplots(figsize=(14, 6))

frames_x = range(1, len(fps_gpu_video) + 1)
ax.plot(frames_x, fps_gpu_video, color=colors_gpu, linewidth=2, alpha=0.8, label='GPU (CUDA)')
ax.plot(range(1, len(fps_cpu_video) + 1), fps_cpu_video, color=colors_cpu, linewidth=2, alpha=0.8, label='CPU')

# Líneas promedio
ax.axhline(y=np.mean(fps_gpu_video), color=colors_gpu, linestyle='--', alpha=0.5, label=f'GPU avg: {np.mean(fps_gpu_video):.1f} FPS')
ax.axhline(y=np.mean(fps_cpu_video), color=colors_cpu, linestyle='--', alpha=0.5, label=f'CPU avg: {np.mean(fps_cpu_video):.1f} FPS')

ax.set_xlabel('Frame', fontsize=12)
ax.set_ylabel('Cuadros por Segundo (FPS)', fontsize=12)
ax.set_title('FPS durante Inferencia en Video: GPU vs CPU', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fps_video_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica guardada en fps_video_comparison.png")

## 8. Script para Ejecución en el Laboratorio (Webcam Real)

El siguiente script está diseñado para ejecutarse **en las computadoras del laboratorio de Cómputo 8** con webcam.
Copia este código y ejecútalo directamente en el lab.

In [ ]:
# ==============================================================================
# 8.1 Script standalone para webcam en el laboratorio
# ==============================================================================
# Guardar este script como archivo .py para ejecutar en el lab

lab_script = '''
#!/usr/bin/env python3
"""
Práctica 4 - Parte 1A: Inferencia en tiempo real con webcam
Segmentación de frutas (Apple, Banana, Orange) con YOLOv11

Ejecutar en el laboratorio de Cómputo 8 con GPU NVIDIA:
    python3 webcam_fruit_seg.py --device gpu
    python3 webcam_fruit_seg.py --device cpu

Autor: [Tu Nombre]
Asignatura: Visión por Computador
"""

import cv2
import time
import argparse
import numpy as np
import psutil
import subprocess
import uuid

try:
    import torch
    from ultralytics import YOLO
except ImportError:
    print("Instalar dependencias: pip install ultralytics torch")
    exit(1)


def get_mac_address():
    """Obtiene la MAC Address del computador."""
    mac = uuid.getnode()
    mac_str = ':'.join(f'{(mac >> (8 * i)) & 0xff:02x}' for i in reversed(range(6)))
    return mac_str


def get_gpu_memory_usage():
    """Obtiene el uso de memoria GPU con nvidia-smi."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True
        )
        used, total = result.stdout.strip().split(",")
        return f"{used.strip()} / {total.strip()} MB"
    except Exception:
        return "N/A"


def main():
    parser = argparse.ArgumentParser(description="Fruit Segmentation - Webcam")
    parser.add_argument("--device", type=str, default="gpu", choices=["gpu", "cpu"],
                        help="Dispositivo para inferencia: gpu o cpu")
    parser.add_argument("--model", type=str, default="best.pt",
                        help="Ruta al modelo entrenado (.pt)")
    parser.add_argument("--conf", type=float, default=0.5,
                        help="Umbral de confianza")
    parser.add_argument("--camera", type=int, default=0,
                        help="Índice de la cámara (0 = default)")
    args = parser.parse_args()

    device = 0 if args.device == "gpu" and torch.cuda.is_available() else "cpu"
    device_name = "GPU (CUDA)" if device != "cpu" else "CPU"

    print(f"\n{'='*60}")
    print(f"  Práctica 4 - Segmentación de Frutas")
    print(f"  Dispositivo: {device_name}")
    print(f"  MAC Address: {get_mac_address()}")
    if device != "cpu":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {get_gpu_memory_usage()}")
    print(f"{'='*60}\n")

    # Cargar modelo
    model = YOLO(args.model)
    print(f"Modelo cargado: {args.model}")

    # Abrir webcam
    cap = cv2.VideoCapture(args.camera)
    if not cap.isOpened():
        print("Error: No se pudo abrir la cámara")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    fps_history = []
    mac_address = get_mac_address()

    print("Presiona 'q' para salir, 's' para guardar screenshot")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Inferencia
        t_start = time.perf_counter()
        results = model.predict(source=frame, device=device, verbose=False, conf=args.conf)
        t_end = time.perf_counter()

        inference_time = t_end - t_start
        current_fps = 1.0 / inference_time if inference_time > 0 else 0
        fps_history.append(current_fps)

        # Dibujar resultado
        annotated = results[0].plot()

        # Overlay de información
        overlay = annotated.copy()
        cv2.rectangle(overlay, (10, 10), (420, 180), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, annotated, 0.4, 0, annotated)

        font = cv2.FONT_HERSHEY_SIMPLEX
        color_val = (0, 255, 0) if device != "cpu" else (0, 128, 255)

        cv2.putText(annotated, f"Device: {device_name}", (20, 35), font, 0.6, (0, 255, 255), 2)
        cv2.putText(annotated, f"FPS: {current_fps:.1f}", (20, 65), font, 0.8, color_val, 2)
        cv2.putText(annotated, f"Inference: {inference_time*1000:.1f} ms", (20, 95), font, 0.6, color_val, 2)

        ram_usage = psutil.Process().memory_info().rss / 1e6
        cv2.putText(annotated, f"RAM: {ram_usage:.0f} MB", (20, 125), font, 0.6, (255, 255, 255), 2)

        if device != "cpu":
            gpu_mem = get_gpu_memory_usage()
            cv2.putText(annotated, f"VRAM: {gpu_mem}", (20, 155), font, 0.6, (255, 255, 255), 2)

        cv2.putText(annotated, f"MAC: {mac_address}", (20, 175 if device != "cpu" else 155),
                    font, 0.5, (200, 200, 200), 1)

        n_dets = len(results[0].boxes) if results[0].boxes is not None else 0
        cv2.putText(annotated, f"Detections: {n_dets}",
                    (annotated.shape[1] - 200, 35), font, 0.6, (255, 255, 0), 2)

        cv2.imshow(f"Fruit Segmentation - {device_name}", annotated)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("s"):
            filename = f"screenshot_{device_name.lower().replace(' ', '_')}_{time.strftime('%H%M%S')}.png"
            cv2.imwrite(filename, annotated)
            print(f"Screenshot guardado: {filename}")

    cap.release()
    cv2.destroyAllWindows()

    # Resumen final
    if fps_history:
        print(f"\n{'='*60}")
        print(f"  RESUMEN DE SESIÓN")
        print(f"  Dispositivo: {device_name}")
        print(f"  Frames procesados: {len(fps_history)}")
        print(f"  FPS promedio: {np.mean(fps_history):.2f}")
        print(f"  FPS mínimo:   {np.min(fps_history):.2f}")
        print(f"  FPS máximo:   {np.max(fps_history):.2f}")
        print(f"  MAC Address:  {mac_address}")
        print(f"{'='*60}")


if __name__ == "__main__":
    main()
'''

# Guardar el script
with open('webcam_fruit_seg.py', 'w') as f:
    f.write(lab_script)

print("✅ Script para laboratorio guardado como: webcam_fruit_seg.py")
print("\nUso en el laboratorio:")
print("  python3 webcam_fruit_seg.py --device gpu --model best.pt")
print("  python3 webcam_fruit_seg.py --device cpu --model best.pt")

## 9. Exportar Modelo y Archivos Necesarios

Exportamos todo lo necesario para llevar al laboratorio.

In [ ]:
# ==============================================================================
# 9.1 Exportar modelo y archivos finales
# ==============================================================================
import shutil

# Crear directorio de exportación
export_dir = Path('practica4_1A_export')
export_dir.mkdir(exist_ok=True)

# Copiar modelo entrenado
shutil.copy(best_model_path, export_dir / 'best.pt')
print(f"✅ Modelo copiado a {export_dir / 'best.pt'}")

# Copiar script de webcam
shutil.copy('webcam_fruit_seg.py', export_dir / 'webcam_fruit_seg.py')
print(f"✅ Script copiado a {export_dir / 'webcam_fruit_seg.py'}")

# Copiar gráficas y evidencias
for img_file in ['dataset_samples.png', 'inference_results.png', 
                 'gpu_vs_cpu_comparison.png', 'fps_video_comparison.png',
                 'validation_comparison.png', 'confusion_matrices.png']:
    if os.path.exists(img_file):
        shutil.copy(img_file, export_dir / img_file)

# Copiar videos de salida
for vid_file in ['output_gpu.mp4', 'output_cpu.mp4']:
    if os.path.exists(vid_file):
        shutil.copy(vid_file, export_dir / vid_file)

print(f"\n📦 Todo exportado en: {export_dir}/")
print("\nContenido:")
for f in sorted(export_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"   📄 {f.name} ({size_mb:.1f} MB)")

In [ ]:
# ==============================================================================
# 9.2 Comprimir para descarga (si estás en Colab)
# ==============================================================================
!zip -r practica4_1A_export.zip practica4_1A_export/

# Si estás en Google Colab, descargar automáticamente
try:
    from google.colab import files
    files.download('practica4_1A_export.zip')
    print("✅ Descarga iniciada")
except ImportError:
    print("No estás en Colab. El archivo está en: practica4_1A_export.zip")

## 10. Conclusiones

### Resultados obtenidos:
- Se realizó **transfer learning** exitoso sobre el modelo YOLOv11 para segmentar 3 clases de frutas (Apple, Banana, Orange)
- El modelo alcanzó métricas satisfactorias de mAP en el conjunto de validación
- La comparación GPU vs CPU demuestra una mejora significativa en rendimiento al usar la GPU

### Observaciones sobre GPU vs CPU:
- La GPU permite procesar más cuadros por segundo, habilitando la inferencia en tiempo real
- El uso de VRAM es relativamente bajo con modelos nano (YOLOv11n-seg)
- En CPU, la inferencia es funcional pero no alcanza tasas de FPS suficientes para aplicaciones en tiempo real

### Herramientas y referencias:
- **Ultralytics YOLO**: https://github.com/ultralytics/ultralytics
- **Roboflow**: https://roboflow.com/
- Jocher, G., Chaurasia, A., & Qiu, J. (2023). Ultralytics YOLO. https://github.com/ultralytics/ultralytics

---
*Práctica desarrollada para la asignatura de Visión por Computador - UPS*